# 03. 개선: 3단 구조 템플릿과 관심사 MCP Tool (논문 5장, 표 4, 그림 3·4)

| 단계 | 입력 | 생성되는 데이터 |
|---|---|---|
| 개선 학습 데이터 | 안심 → 칭찬/격려 → 참여 유도 3단 구조 템플릿 | 3단 구조 instruction 데이터 |
| LoRA 재학습 | 3단 구조 데이터 | 병합 모델 `gemma-LoRA` (Google Drive에 저장) |
| 관심사 검색 | 좋아하는 아티스트(예: `아이유 에스파 방탄소년단`) | 아티스트별 YouTube 영상 목록 |
| 메시지 생성 | 관심사 영상 제목 + 완료 여부 + 인증 정보 + 변경 프롬프트 | 20단어 이내 3단 구조 메시지 |

메시지 생성 단계는 `02_message_pipeline.ipynb`의 입력(`user_command`), 판별 결과(`result_attendence`), 생성 함수(`gemma_generate`)를 사용합니다.
원본에서는 02와 하나의 노트북에서 이어서 실행했으므로, 02를 실행한 런타임에서 이어서 실행합니다.

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes #라이브러리 설치
!pip install -q datasets transformers peft accelerate bitsandbytes scikit-learn


## 1. 3단 구조 학습 데이터 생성

In [2]:
import random
from datasets import Dataset

# ✅ 완료 학생 템플릿 (칭찬 50개, 3단 구조)
praise_templates = [
    "오늘도 멋진 하루였어! 에스파 'Whiplash'처럼, 7회차 인증 완료! 너의 집중력이 빛났어 🎯 내일 수업에서도 너의 에너지를 보여줘!",
    "너의 노력이 빛나고 있어! 아이유 'Celebrity'처럼, 숙제 챌린지 10일 연속 클리어! 정말 자랑스러워. 다음 숙제에도 멋지게 도전하자!",
    "오늘은 정말 대단했어! BTS 'Dynamite'처럼, 숙제 챌린지 5일차 성공! 오늘 최고야 ✨ 내일도 수업에서 활기차게 참여해!",
    "밝은 에너지가 전해져! IVE 'I AM'처럼, 7회차 인증 완료! 스스로를 믿고 끝냈네. 다음 수업에서도 자신감 있게 보여줘!",
    "너의 성취가 눈부셔! 아이유 'Love wins all'처럼, 숙제 챌린지 10일 연속 클리어! 끝까지 해냈어. 내일도 교실에서 빛나자!",
    "정말 기분 좋은 하루야! BTS 'Boy With Luv'처럼, 숙제 챌린지 5일차 성공! 즐겁게 완성했네. 다음 숙제에도 도전해봐!",
    "너의 하루가 특별해! 아이유 'Blueming'처럼, 7회차 인증 완료! 숙제가 꽃피웠네 🌸 내일 수업에서도 활짝 웃자!",
    "집중력이 대단했어! 에스파 플레이리스트처럼, 숙제 챌린지 10일 연속 클리어! 멋져. 다음 수업에서도 활기차게 참여하자!",
    "오늘 네 모습이 인상적이야! BTS 'Go Go'처럼, 숙제 챌린지 5일차 성공! 파워풀했어. 내일도 교실에서 힘내자!",
    "너의 성장은 확실해! 아이유 콘서트처럼, 7회차 인증 완료! 숙제를 무대로 만들었네. 다음에도 더 멋지게 보여줘!",
    "오늘도 최고였어! BTS 'DNA'처럼, 숙제 챌린지 10일 연속 클리어! 집중력이 흔들림 없었어. 내일도 이어가자!",
    "너의 밝음이 돋보여! 에스파 신곡처럼, 숙제 챌린지 5일차 성공! 새롭게 마무리했네. 수업에서도 활짝 빛나자!",
    "너의 색이 뚜렷해! 아이유 'Palette'처럼, 7회차 인증 완료! 숙제를 네 방식으로 끝냈네. 내일 수업도 기대돼!",
    "정말 멋진 하루야! BTS 'Butter'처럼, 숙제 챌린지 10일 연속 클리어! 매끄럽게 완성했네. 내일도 즐겁게 하자!",
    "오늘도 대단했어! 에스파 'Next Level'처럼, 숙제 챌린지 5일차 성공! 한 단계 성장했네. 내일 교실에서도 보여줘!",
    "너의 성취가 눈부셔! 아이유 '좋은 날'처럼, 7회차 인증 완료! 오늘이 더 빛나네. 내일도 이어가자!",
    "오늘 집중력이 굉장했어! BTS 'Run BTS'처럼, 숙제 챌린지 10일 연속 클리어! 멈추지 않았네. 내일도 도전하자!",
    "너의 에너지가 강렬해! 에스파 노래처럼, 숙제 챌린지 5일차 성공! 멋지게 끝냈어. 교실에서도 활기차게 하자!",
    "밝은 성취야! 아이유 '라일락'처럼, 7회차 인증 완료! 화사하게 숙제를 끝냈네. 내일도 밝게 시작하자!",
    "정말 뜨거웠어! BTS 'Fire'처럼, 숙제 챌린지 10일 연속 클리어! 불태웠네 🔥 내일도 열정 보여줘!",
    "너의 성취가 감동적이야! 에스파 'Dreams Come True'처럼, 숙제 챌린지 5일차 성공! 꿈같은 하루야. 내일도 함께하자!",
    "너의 정리가 깔끔해! 아이유 '에잇'처럼, 7회차 인증 완료! 완벽히 마무리했네. 수업에서도 보여줘!",
    "오늘도 멋졌어! BTS 'MIC Drop'처럼, 숙제 챌린지 10일 연속 클리어! 당당히 끝냈어. 내일도 도전해!",
    "너의 집중이 대단해! 에스파 'Drama'처럼, 숙제 챌린지 5일차 성공! 인상 깊게 끝냈네. 수업에서도 멋지게 해보자!",
    "정말 유쾌했어! 아이유 '스물셋'처럼, 7회차 인증 완료! 즐겁게 숙제를 마무리했네. 내일도 활기차게!",
    "차분하게 해냈어! BTS 'Life Goes On'처럼, 숙제 챌린지 10일 연속 클리어! 자연스럽게 이어갔네. 내일도 이어가자!",
    "너의 힘이 느껴져! 에스파 'Black Mamba'처럼, 숙제 챌린지 5일차 성공! 강렬하게 끝냈어. 내일도 화이팅!",
    "오늘도 따뜻했어! 아이유 '너랑 나'처럼, 7회차 인증 완료! 숙제를 따뜻하게 마무리했네. 교실에서도 이어가자!",
    "너의 성취가 따뜻해! BTS 'Spring Day'처럼, 숙제 챌린지 10일 연속 클리어! 감동적이야. 내일도 함께하자!",
    "정말 파워풀했어! 에스파 'Savage'처럼, 숙제 챌린지 5일차 성공! 멋진 집중력이었어. 수업에서도 이어가자!",
    "오늘도 따뜻했어! 아이유 '밤편지'처럼, 7회차 인증 완료! 부드럽게 끝냈네. 내일도 도전해!",
    "너의 끈기가 대단해! BTS 'Not Today'처럼, 숙제 챌린지 10일 연속 클리어! 끝까지 포기 안 했네. 내일도 화이팅!",
    "오늘 창의적이었어! 에스파 'Illusion'처럼, 숙제 챌린지 5일차 성공! 특별했어. 내일도 멋지게!",
    "즐거운 하루였어! 아이유 '금요일에 만나요'처럼, 7회차 인증 완료! 숙제를 기분 좋게 끝냈네. 교실에서도 빛나자!",
    "행복한 성취야! BTS 'Euphoria'처럼, 숙제 챌린지 10일 연속 클리어! 웃음 가득했네. 내일도 이어가자!",
    "오늘도 멋졌어! 에스파 'Better Things'처럼, 숙제 챌린지 5일차 성공! 좋은 결과 남겼네. 수업에서도 도전해!",
    "너의 끈기가 빛나! 아이유 '내 손을 잡아'처럼, 7회차 인증 완료! 끝까지 붙잡았네. 내일도 함께하자!",
    "오늘도 당당했어! BTS 'IDOL'처럼, 숙제 챌린지 10일 연속 클리어! 자랑스러워. 교실에서도 보여줘!",
    "열정이 대단해! 에스파 'Thirsty'처럼, 숙제 챌린지 5일차 성공! 뜨겁게 해냈네. 내일도 이어가자!",
    "오늘 감동적이야! 아이유 '드라마'처럼, 7회차 인증 완료! 숙제를 끝냈네. 수업에서도 멋지게!",
    "집중력이 최고였어! BTS 'Save Me'처럼, 숙제 챌린지 10일 연속 클리어! 대단했어. 내일도 이어가자!",
    "끝까지 해냈어! 에스파 'Hold On Tight'처럼, 숙제 챌린지 5일차 성공! 집중했네. 내일도 보여줘!",
    "오늘도 성취했어! 아이유 '좋은 날'처럼, 7회차 인증 완료! 웃음 가득했네. 내일도 이어가자!",
    "너의 힘이 대단해! BTS 'ON'처럼, 숙제 챌린지 10일 연속 클리어! 파워풀했네. 내일도 도전하자!",
    "정말 열정적이야! 에스파 'Spicy'처럼, 숙제 챌린지 5일차 성공! 뜨겁게 끝냈네. 교실에서도 활기차게!",
    "특별했어! 아이유 'Celebrity'처럼, 7회차 인증 완료! 숙제를 완성했네. 내일도 보여줘!",
    "오늘도 멋졌어! BTS 'DOPE'처럼, 숙제 챌린지 10일 연속 클리어! 멋지게 끝냈네. 내일도 도전해!",
    "완벽한 하루야! 에스파 'Life’s Too Short'처럼, 숙제 챌린지 5일차 성공! 완벽 클리어! 내일도 이어가자!",
    "너의 무대였어! 아이유 콘서트처럼, 7회차 인증 완료! 숙제를 소화했네. 수업에서도 빛나자!",
    "오늘도 즐거웠어! BTS 'Permission to Dance'처럼, 숙제 챌린지 10일 연속 클리어! 행복했어. 내일도 함께하자!"
]

# ✅ 미완료 학생 템플릿 (격려 50개, 3단 구조)
encouragement_templates = [
    "오늘 다 못했어도 괜찮아. 에스파 'Whiplash'처럼, 숙제 부담 휙 날리고 다시 화이팅! 넌 할 수 있어. 내일 수업에서 다시 도전하자!",
    "실수도 성장의 일부야. 아이유 'Celebrity'처럼, 너만의 방식으로 숙제 다시 도전! 힘내자. 다음 수업에서 네 색깔을 보여줘!",
    "조금 늦어도 괜찮아. BTS 'Dynamite'처럼, 내일은 밝게 숙제를 폭발시켜! 파워 업! 교실에서도 멋지게 해보자!",
    "포기하지 않아도 돼. IVE 'I AM'처럼, 스스로를 믿고 숙제를 다시 해보자! 넌 이미 멋져. 내일 참여로 증명해보자!",
    "오늘 힘들었지? 아이유 'Love wins all'처럼, 노력은 반드시 빛날 거야. 다시 파이팅! 내일 교실에서 활짝 빛나자!",
    "쉬어가도 괜찮아. BTS 'Boy With Luv'처럼, 웃으며 숙제에 다시 달려가자! 화이팅! 다음 수업에서도 힘내자!",
    "네 가치는 여전해. 아이유 'Blueming'처럼, 내일은 더 멋진 성취 피울 수 있어 🌸 수업에서도 밝게 시작하자!",
    "천천히 해도 돼. 에스파 플레이리스트처럼, 차근차근 숙제를 완성할 수 있어. 내일은 수업에서 한 걸음 더 가자!",
    "실패는 과정일 뿐이야. BTS 'Go Go'처럼, 에너지 넘치게 숙제 다시 시작하자! 힘내! 교실에서 더 멋지게 보여줘!",
    "너는 충분히 잘하고 있어. 아이유 콘서트처럼, 숙제 무대는 아직 준비 중이야. 넌 할 수 있어. 내일은 참여로 이어가자!",
    "오늘도 괜찮아. BTS 'DNA'처럼, 내일은 흔들림 없이 숙제를 이어가자! 자신감을 가져. 수업에서도 힘내자!",
    "불안해하지 마. 에스파 신곡처럼, 새롭게 시작하면 숙제도 완성할 수 있어. 내일 교실에서 다시 도전하자!",
    "조금씩 해도 돼. 아이유 'Palette'처럼, 숙제에 네 색을 더해보자. 내일도 교실에서 보여줘!",
    "완벽하지 않아도 돼. BTS 'Butter'처럼, 부드럽게 숙제를 이어가자. 넌 충분히 할 수 있어. 교실에서도 화이팅!",
    "실패도 배움이야. 에스파 'Next Level'처럼, 이번엔 레벨업의 기회야! 내일 수업에서 증명하자!",
    "오늘 조금 힘들었지? 아이유 '좋은 날'처럼, 내일은 반드시 더 나아질 거야. 교실에서 활짝 웃자!",
    "쉬어가도 괜찮아. BTS 'Run BTS'처럼, 멈추지 말고 숙제 끝까지 달려가자! 수업에서도 힘내자!",
    "아직 기회는 있어. 에스파 노래처럼, 오늘은 놓쳤지만 내일은 반드시 화이팅! 교실에서도 함께하자!",
    "네 가능성은 충분해. 아이유 '라일락'처럼, 다시 활짝 시작해보자. 수업에서도 웃으며 참여하자!",
    "오늘 놓쳤어도 괜찮아. BTS 'Fire'처럼, 뜨겁게 숙제를 불태우자! 교실에서 도전 이어가자!",
    "다시 시작하면 돼. 에스파 'Dreams Come True'처럼, 숙제도 반드시 완성할 수 있어. 내일도 교실에서 도전해!",
    "서두르지 않아도 돼. 아이유 '에잇'처럼, 가볍게라도 숙제를 다시 시작해봐. 교실에서 이어가자!",
    "포기하지 마. BTS 'MIC Drop'처럼, 당당히 숙제를 붙잡자! 내일 교실에서 멋지게 하자!",
    "괜찮아, 아직 기회 있어. 에스파 'Drama'처럼, 멋진 숙제 이야기를 완성하자! 교실에서도 이어가자!",
    "유쾌하게 해도 돼. 아이유 '스물셋'처럼, 즐겁게 숙제를 다시 해보자! 수업에서도 웃으며 참여하자!",
    "천천히 이어가자. BTS 'Life Goes On'처럼, 차근차근 숙제를 해내면 돼. 교실에서도 보여줘!",
    "오늘 못했어도 괜찮아. 에스파 'Black Mamba'처럼, 집중력을 다시 불태워! 내일 교실에서 멋지게 해보자!",
    "너는 충분히 소중해. 아이유 '너랑 나'처럼, 함께라면 숙제도 곧 끝나. 내일 수업에서 함께하자!",
    "조금 힘들었지? BTS 'Spring Day'처럼, 따뜻하게 힘내서 해내자! 교실에서 도전 이어가자!",
    "강하게 도전하자! 에스파 'Savage'처럼, 이번엔 반드시 해낼 거야. 내일 교실에서 화이팅!",
    "편히 쉬어도 돼. 아이유 '밤편지'처럼, 내일은 따뜻하게 시작하자. 교실에서도 이어가자!",
    "오늘은 실패했어도 괜찮아. BTS 'Not Today'처럼, 내일은 반드시 해낼 거야! 교실에서 도전 이어가자!",
    "새롭게 시작하자. 에스파 'Illusion'처럼, 창의적으로 숙제를 다시 해보자! 내일 교실에서도 보여줘!",
    "즐겁게 해도 돼. 아이유 '금요일에 만나요'처럼, 기분 좋게 숙제를 해내자. 교실에서도 이어가자!",
    "행복은 올 거야. BTS 'Euphoria'처럼, 숙제를 완성할 순간이 곧 와. 내일 수업에서 이어가자!",
    "더 나은 내일 있어. 에스파 'Better Things'처럼, 좋은 결과가 곧 와! 교실에서도 화이팅!",
    "용기 내도 돼. 아이유 '내 손을 잡아'처럼, 다시 시작해보자. 내일 교실에서도 이어가자!",
    "자신감 가져! BTS 'IDOL'처럼, 숙제에 당당히 맞서자! 내일 교실에서 화이팅!",
    "열정은 충분해. 에스파 'Thirsty'처럼, 가득한 마음으로 숙제를 이어가자! 교실에서도 보여줘!",
    "끝까지 해보자. 아이유 '드라마'처럼, 네 이야기를 완성하자! 교실에서도 멋지게 이어가자!",
    "다시 일어나자. BTS 'Save Me'처럼, 힘내서 숙제를 해내자! 내일 교실에서도 화이팅!",
    "끝까지 붙잡자. 에스파 'Hold On Tight'처럼, 숙제를 완성하자! 교실에서도 도전해보자!",
    "내일은 웃자. 아이유 '좋은 날'처럼, 분명 더 나아질 거야! 교실에서도 밝게 시작하자!",
    "강하게 도전하자! BTS 'ON'처럼, 파워풀하게 숙제를 이어가자! 교실에서도 보여줘!",
    "오늘 놓쳤어도 괜찮아. 에스파 'Spicy'처럼, 열정 넘치게 숙제를 해보자! 교실에서도 화이팅!",
    "특별한 너야. 아이유 'Celebrity'처럼, 숙제를 네 방식으로 해내자. 내일 교실에서도 도전하자!",
    "다시 시작할 수 있어. BTS 'DOPE'처럼, 멋지게 숙제를 완성하자! 교실에서도 보여줘!",
    "놓쳐도 괜찮아. 에스파 'Life’s Too Short'처럼, 내일은 반드시 해낼 수 있어! 교실에서도 이어가자!",
    "결국 노력은 빛나. 아이유 'Love wins all'처럼, 포기하지 말자! 내일 교실에서도 화이팅!",
    "오늘도 괜찮아. BTS 'Permission to Dance'처럼, 즐겁게 숙제를 해내자! 내일 교실에서 웃자!"
]

# ✅ 250쌍씩 praise & encouragement 생성
praise_data = [
    {
        "type": "attendance",
        "content": f"이번 챌린지 {i%30+1}회차 인증 (누적 {i%30}회 인증)",
        "complete": "1",
        "response": random.choice(praise_templates)
    }
    for i in range(500)
]

encouragement_data = [
    {
        "type": "attendance",
        "content": f"이번 챌린지 {i%30+1}회차 인증 (누적 {i%30}회 인증)",
        "complete": "0",
        "response": random.choice(encouragement_templates)
    }
    for i in range(500)
]

# ✅ 데이터 합치고 섞기
combined_data = praise_data + encouragement_data
random.shuffle(combined_data)

# ✅ 프롬프트 + 응답 형식으로 변환
instruction_dataset = [
    {
        # input: 상황 + 성과 멘트 강조
        "input": (
            f"{'숙제를 완료한' if d['complete'] == '1' else '숙제를 완료하지 못한'} 아이에게 "
            f"메세지 생성: {d['content']} "
            f"{'(예: 숙제 챌린지 10일 연속 클리어!, 7회차 인증 완료!)' if d['complete'] == '1' else '(예: 다시 7회차 도전!, 내일은 반드시 화이팅!)'}"
        ),
        # output: 실제 응답
        "output": d["response"]
    }
    for d in combined_data
]

# ✅ Dataset 객체 생성
dataset = Dataset.from_list(instruction_dataset)

# 결과 확인 (선택적 출력)
print(f"총 샘플 수: {len(dataset)}")
print("예시:\n", dataset[0])
# 🤖 Dataset 생성
dataset = Dataset.from_list(instruction_dataset)

print(dataset.column_names)

총 샘플 수: 1000
예시:
 {'input': '숙제를 완료한 아이에게 메세지 생성: 이번 챌린지 16회차 인증 (누적 15회 인증) (예: 숙제 챌린지 10일 연속 클리어!, 7회차 인증 완료!)', 'output': '너의 밝음이 돋보여! 에스파 신곡처럼, 숙제 챌린지 5일차 성공! 새롭게 마무리했네. 수업에서도 활짝 빛나자!'}
['input', 'output']


## 2. LoRA 재학습과 병합 모델 저장

In [3]:
#gemma 3B 짜리
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from pathlib import Path
import torch
# 로컬 디렉토리 경로
model_path = Path("/content/drive/MyDrive/gemma-3n-E2b-it")  # 문자열 또는 Path 객체

device = "cuda" if torch.cuda.is_available() else "cpu"
# Tokenizer와 Model을 로컬에서 불러옴
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32, # <-- Change to torch.float16
    device_map="auto",
    local_files_only=True
)

# LoRA 설정
lora_config = LoraConfig(
    r=16, # LoRA의 핵심 파라미터, 저랭크(rank) 차원 수를 의미, A,B 배열 크기
    lora_alpha=32, # 학습 효과를 조정하는 확대/축소 계수 alpha/lora = 2 2배 더 크게 반영
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5, #학습 시 일부 업데이트를 랜덤으로 꺼서 과적합을 방지
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# PEFT 모델 구성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 4,374,528 || all params: 5,443,812,800 || trainable%: 0.0804


In [4]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, default_data_collator
from transformers import AutoTokenizer, BitsAndBytesConfig
import torch

# 🧠 템플릿 포맷 함수
special_tokens = {
    "additional_special_tokens": ["<start_of_turn>", "<end_of_turn>"]
}
tokenizer.add_special_tokens(special_tokens)              # 토크나이저에 추가
model.resize_token_embeddings(len(tokenizer))             # 모델 임베딩 크기 재조정

# ✅ 템플릿 포맷 함수 (instruction → text 하나로 합침)
def format_instruction(example):
    return {
        "text": f"<start_of_turn>user\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    }

formatted_dataset = dataset.map(format_instruction)

# 🧼 Tokenize 함수
def tokenize(example):
    tokenized = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128  # 너무 짧으면 안 됨
    )
    input_ids = tokenized["input_ids"]

    # <start_of_turn>model 이후만 label로 사용
    model_token = tokenizer.convert_tokens_to_ids("<start_of_turn>")
    model_index = input_ids.index(model_token) if model_token in input_ids else 0

    labels = [-100] * model_index + input_ids[model_index:]
    tokenized["labels"] = labels
    return tokenized

# 🧪 Tokenize 적용
tokenized_dataset = formatted_dataset.map(tokenize, remove_columns=formatted_dataset.column_names)

# ⚙️ 학습 설정
training_args = TrainingArguments(
    output_dir="./results",                     # ✅ 모델 학습 결과(가중치, 로그 등)를 저장할 디렉토리 경로
    per_device_train_batch_size=1,              # ✅ 각 디바이스(GPU/CPU)당 처리할 배치 사이즈 (작은 모델일수록 작게)
    gradient_accumulation_steps=4,              # ✅ 실제 배치 사이즈를 키우기 위한 누적 단계 수
                                                #    → 실 배치 사이즈 = 1 × 4 = 4
    num_train_epochs=4,                         # ✅ 학습 데이터를 몇 번 반복해서 학습할지 (에폭 수)
    logging_steps=50,                            # ✅ 학습 로그를 출력할 간격 (스텝 단위)
    save_steps=100,                              # ✅ 모델을 저장할 간격 (스텝 단위)
    fp16=True,                                  # ✅ 16-bit float 연산을 사용해 메모리 사용량과 속도를 최적화
                                                #    (단, GPU가 필요하고 지원돼야 함)
    report_to="none"                            # ✅ 로깅 툴 사용 안 함 (예: wandb, tensorboard 사용 안 함)
)

# 📦 모델 준비
model.train()
model.config.use_cache = False
model.print_trainable_parameters()  # ✅ LoRA 파라미터만 학습되는지 확인

# ⚙️ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

# 🚀 학습 시작
trainer.train()

trainable params: 4,374,528 || all params: 5,443,812,800 || trainable%: 0.0804


Step,Training Loss
50,18.407600
100,3.504600
150,2.910800
200,2.542800
250,2.298200
300,2.152900
350,2.034300
400,1.866300
450,1.776300
500,1.710100


TrainOutput(global_step=1000, training_loss=2.6185758323669432, metrics={'train_runtime': 1662.4281, 'train_samples_per_second': 2.406, 'train_steps_per_second': 0.602, 'total_flos': 8886352674816000.0, 'train_loss': 2.6185758323669432, 'epoch': 4.0})

In [5]:
merged = model.merge_and_unload()  # LoRA를 base에 병합 이후 원하는 드라이브 장소에 모델 저장(나중에 바로 사용 가능)
merge_dir = "/content/drive/MyDrive/gemma-LoRA"
merged.save_pretrained(merge_dir, safe_serialization=True)
tokenizer.save_pretrained(merge_dir)

('/content/drive/MyDrive/gemma-LoRA/tokenizer_config.json',
 '/content/drive/MyDrive/gemma-LoRA/special_tokens_map.json',
 '/content/drive/MyDrive/gemma-LoRA/chat_template.jinja',
 '/content/drive/MyDrive/gemma-LoRA/tokenizer.model',
 '/content/drive/MyDrive/gemma-LoRA/added_tokens.json',
 '/content/drive/MyDrive/gemma-LoRA/tokenizer.json')

## 3. 관심사 기반 콘텐츠 수집 (관심사 MCP Tool)

입력한 아티스트마다 YouTube 검색 API로 영상을 찾아 `final_combined_results`로 합칩니다.

In [ ]:
# 여러명 입력은 띄어쓰기 한 번씩 이용해서 입력하세요
# ex) 에스파 아이브 아이유
interest = input("좋아하는 아티스트를 입력하세요: ")
interest_list = list(interest.split(' '))

좋아하는 아티스트를 입력하세요: 아이유 에스파 방탄소년단


In [ ]:
## 좋아하는 가수 노래 검색(추가 tool로 사용)
import os
import requests
import json  # 🔹 추가
from google.colab import userdata
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API')
YOUTUBE_URL = "https://www.googleapis.com/youtube/v3/search"

def get_artist_music_videos(artist_name: str, max_results: int = 50, order_by: str = "relevance"):
    if not YOUTUBE_API_KEY:
        return {
            "status": "error",
            "message": "❌ YOUTUBE_API_KEY가 설정되지 않았습니다."
        }

    all_results = []
    next_page_token = None
    current_results_count = 0

    # 검색 API는 한 번에 최대 50개 결과를 반환합니다.
    api_max_results_per_call = 50

    try:
        while current_results_count < max_results:
            params = {
                "part": "snippet",  # 검색 API는 statistics를 직접 제공하지 않을 수 있습니다.
                "q": f"{artist_name} 노래", # <<< 가수의 이름을 검색어로 사용
                "type": "video",    # <<< 동영상만 검색
                "regionCode": "KR",
                "maxResults": api_max_results_per_call,
                "key": YOUTUBE_API_KEY,
                "order": order_by # 정렬 기준 (relevance, viewCount 등)
            }

            if next_page_token:
                params["pageToken"] = next_page_token

            # 검색 엔드포인트 사용
            res = requests.get(YOUTUBE_URL, params=params) # <<< Youtube_URL 사용

            if res.status_code != 200:
                return {
                    "status": "error",
                    "message": f"❌ API 요청 실패: {res.text}"
                }

            data = res.json()
            items = data.get("items", [])

            if not items:
                break

            # 'get_trending_music_videos'에서 사용한 뮤직 키워드 필터링을
            # 여기서는 적용하지 않거나, 더 좁게 적용할 수 있습니다.
            # 이미 가수 이름으로 검색했으므로 제목에 '뮤직' 같은 키워드가 없어도 됩니다.
            # 하지만 혹시 모를 오차를 위해 가볍게 필터링할 수도 있습니다.
            # 여기서는 제목에 아티스트 이름이 포함된 경우를 선호하여 필터링합니다.

            for item in items:
                snippet = item["snippet"]
                video_id = item["id"].get("videoId") # 검색 API는 videoId가 id 객체 안에 있습니다.

                # 가수의 이름이 제목이나 채널에 포함된 경우만 추출
                if video_id and \
                   (artist_name.lower() in snippet["title"].lower() or \
                    artist_name.lower() in snippet["channelTitle"].lower()):

                    # 검색 API는 statistics를 바로 제공하지 않으므로,
                    # 조회수 등을 얻으려면 별도로 videos.list를 호출해야 합니다.
                    # 여기서는 간단히 검색 결과만 반환합니다.

                    all_results.append({
                        "title": snippet["title"],
                        "channel": snippet["channelTitle"],
                        "video_url": f"https://www.youtube.com/watch?v={video_id}",
                        "thumbnail_url": snippet["thumbnails"]["high"]["url"],
                        # "view_count": "N/A" # 필요하다면 여기서 videos.list를 추가 호출하여 가져올 수 있습니다.
                    })
                    current_results_count = len(all_results)
                    if current_results_count >= max_results:
                        break

            next_page_token = data.get("nextPageToken")
            if not next_page_token:
                break

        if not all_results:
            return {
                "status": "error",
                "message": f"⚠️ '{artist_name}'의 음악 동영상이 없습니다."
            }

        return {
            "status": "ok",
            "videos": all_results[:max_results]
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

# 모든 아티스트의 동영상을 담을 리스트
all_artist_videos_list = []
# API 호출 성공 여부를 추적할 변수
all_api_calls_ok = True
# 에러 메시지를 담을 리스트
error_messages = []

artists_to_search = interest_list
results_per_artist = 10

for artist_name in artists_to_search:
    # 각 아티스트별로 함수 호출
    artist_result = get_artist_music_videos(artist_name=artist_name, max_results=results_per_artist)

    if artist_result["status"] == "ok":
        # 성공 시, 해당 아티스트의 'videos' 리스트를 전체 리스트에 추가
        all_artist_videos_list.extend(artist_result["videos"])
    else:
        # 실패 시, 오류 메시지를 저장하고 성공 여부 플래그 변경
        all_api_calls_ok = False
        error_messages.append(f"'{artist_name}' 검색 중 오류: {artist_result['message']}")

# 최종 결과를 하나의 딕셔너리 형태로 만듭니다.
# 모든 호출이 성공했으면 status는 'ok', 그렇지 않으면 'error'입니다.
final_combined_results = {
    "status": "ok" if all_api_calls_ok else "error",
    "videos": all_artist_videos_list
}

if not all_api_calls_ok:
    final_combined_results["message"] = "\n".join(error_messages)

print(final_combined_results)

# --- 최종 결과 출력 ---
# 여기서는 3명의 아티스트가 각각 5개씩 가져오므로 최대 15개 동영상이 출력될 수 있습니다.
if final_combined_results["status"] == "ok":
    print(f"\n총 {len(final_combined_results['videos'])}개의 음악 비디오를 찾았습니다.")
    for video in final_combined_results["videos"][:30]: # 최대 15개 출력
        print(f"  제목: {video['title']}")
else:
    print(f"\n오류: {final_combined_results['message']}")

{'status': 'ok', 'videos': [{'title': '아이유 노래모음 [IU / Playlist]', 'channel': '별빛', 'video_url': 'https://www.youtube.com/watch?v=hyRjJG5mq_Q', 'thumbnail_url': 'https://i.ytimg.com/vi/hyRjJG5mq_Q/hqdefault.jpg'}, {'title': '아이유 노래모음 (리메이크 앨범 꽃갈피 셋 가사 포함) + KPOP 플레이리스트 | PLAYLIST 3', 'channel': '탑플뮤직', 'video_url': 'https://www.youtube.com/watch?v=VxyIp8WbkFU', 'thumbnail_url': 'https://i.ytimg.com/vi/VxyIp8WbkFU/hqdefault.jpg'}, {'title': '[ PLAYLIST ] 아이유 노래모음', 'channel': '무말랭이', 'video_url': 'https://www.youtube.com/watch?v=2O2eehHpz2A', 'thumbnail_url': 'https://i.ytimg.com/vi/2O2eehHpz2A/hqdefault.jpg'}, {'title': '♥️아이유 꽃갈피 노래모음ㅣ🌸꽃갈피 셋 리메이크 앨범 전곡 듣기ㅣIU Playlist', 'channel': '마드모아젤', 'video_url': 'https://www.youtube.com/watch?v=t9_nYBev2ls', 'thumbnail_url': 'https://i.ytimg.com/vi/t9_nYBev2ls/hqdefault.jpg'}, {'title': '[𝐏𝐥𝐚𝐲𝐥𝐢𝐬𝐭] 아이유 노래모음', 'channel': '𝐏𝐚𝐬𝐬𝐢𝐨𝐧', 'video_url': 'https://www.youtube.com/watch?v=8jk_iIy2C1Y', 'thumbnail_url': 'https://i.ytimg.com/vi/8jk_iIy2C1Y/hqde

## 4. 메시지 생성 (변경 프롬프트)

변경 프롬프트는 메시지를 (안심) → (칭찬/격려) → (참여 유도) 세 부분으로 구성하고, 20단어 이내 한 줄로 제한합니다.

In [ ]:
#학습된 gemma 불러오기, LoRA 학습 파일에서 모델 학습시키기
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from pathlib import Path
import torch
# 로컬 디렉토리 경로
model_path = Path("/content/drive/MyDrive/gemma-LoRA")  # 학습한 gemma의 경로

device = "cuda" if torch.cuda.is_available() else "cpu"
# Tokenizer와 Model을 로컬에서 불러옴
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    local_files_only=True
)

# LoRA 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# PEFT 모델 구성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 4,374,528 || all params: 5,443,812,800 || trainable%: 0.0804


In [ ]:
# LoRA를 사용하여 학습한 gemma를 사용하여 최종 메시지 생성
import json
import random

user_command_dict = json.loads(user_command)

for i in range (5):
  result = final_combined_results["videos"][i*5]['title']
  followup_prompt = (
        f"<start_of_turn>user\n"
        f"숙제 {result_attendence} 아이에게 1문장 동기부여 메시지 생성\n"
        f"[도구 결과] {result}\n"
        f"[응원 메시지 생성 가이드라인]\n"
        f"1. 도구 결과를 꼭 사용해서 학생의 수업/과제 상황과 자연스럽게 연결해.\n"
        f"2. 메시지는 반드시 세 부분으로 구성해.\n"
        f"   - (안심) 학생의 불안이나 부담을 줄여주는 말\n"
        f"   - (칭찬/격려) 숙제 완료 여부에 따라 성과 칭찬 또는 도전 격려\n"
        f"   - (참여 유도) 내일 수업/과제 참여로 이어지도록 행동을 촉구\n"
        f"3. 자기결정성이론(Self-Determination Theory)에 따라 자율성, 유능감, 관계성을 북돋는 표현을 포함해.\n"
        f"4. 따뜻하고 유머러스한 말투로 작성해.\n"
        f"5. 최종 출력은 20단어 이내의 한 줄 문장으로 작성해.\n"
        f"참고 문맥: {user_command_dict.get('content')}<end_of_turn>\n"
        f"<start_of_turn>model\n"
  )

  final_response = gemma_generate(followup_prompt)
  print(f"최종 응답: {final_response}")

최종 응답: 아이유의Playlist처럼 즐겁게 숙제 마무리했네! 스스로 해냈다는 유능감과 다음 도전을 향한 자율성을 키우자!
최종 응답: Playlist 감명 깊게 들었네! 숙제 마무리도 잘했고, 다음 챌린지도 주도적으로 해봐요!
최종 응답: 🎧 에스파 노래 모음처럼 멋지게 숙제 마무리했네! 스스로 성취감을 느끼고 다음 도전을 향해 나아가길 바라요.
최종 응답: 에스파 에스파의 매력에 빠지다니! 숙제 마무리도 성취감과 함께 해낼 수 있어요. 내일도 새로운 도전을 통해 성장하는 모습이 기대됩니다!
최종 응답: 와, Dynamite처럼 신나는 결과야! 자기 주도적으로 문제를 해결해서 대단해, 내일도 즐겁게 참여하자!
